In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import scipy.stats

# Data exploration: [AMES dataset](http://jse.amstat.org/v19n3/decock.pdf)

## Introduction

In this notebook, we will explore a dataset that we will use again later on during the formation to train models.

This dataset is a good playground because it contains all kinds of variables, each with its set of challenges.

Today, we will follow the following steps:

1. obtain data
2. clean data
3. explore data

## Problem statement

The end goal for our study is to be able to predict prices for the housing market in Iowa, in the US.

In order to have a correctly stated problem, it's required to formulate from the get go what are the expected outputs and how we will evaluate them.

- *Define the model output given the end goal.*

- *Suggest one or several ways to evaluate the quality of the models we will develop later on.*

- *What would be the issue if we delayed the definition of those output and evaluation elements?*

*Your answer here.*

### Solution

- the model output is the house prices
- the difference between what was predicted and the reality, probably using absolute value or squaring
- the introduction of a statistical bias: we could pick the best-performing metric for the model we're evaluating

## Obtaining data

Once the git repository below is cloned, the data should be available in the `dataset-ames/` directory. In particular, it contains a `train.csv` file that you can use as a training set, and a `test.csv` file that can be used to simulate what you have in production: no target is available.

Hints:

- [`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) will be useful. The index column in the data is named `Id`, therefore you can give the `index_col="Id"` argument to this method, in addition to the CSV path.

- Be sure to check out the documentation of [pandas `DataFrame`s](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html).

Exercise:

- *Display the name of the dataframe columns and the shape of the dataframe (number of rows, number of columns).*

- *A description of the dataset variables is available in the file `data_description.txt`. Display it in a cell or download it and study one or two variables*.

In [ ]:
from google.colab import files

!git clone https://github.com/nzmonzmp/dataset-ames.git
!ls -l dataset-ames/
files.download("dataset-ames/data_description.txt")

In [ ]:
# train_df = pandas.???
# test_df = pandas.???

### Solution

In [ ]:
train_df = pandas.read_csv("dataset-ames/train.csv", index_col="Id")
test_df = pandas.read_csv("dataset-ames/test.csv", index_col="Id")

print(f"Columns: {', '.join(train_df.columns)}")
print(f"Shape of the train dataframe: {train_df.shape}")
print(f"Shape of the test dataframe:  {test_df.shape}")

## Extracting variables

*Create the following variables:*

- *`train_X` that contains all the columns of `train_df` except `SalePrice`. You can use indexing or [`DataFrame.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html?highlight=drop#pandas.DataFrame.drop)*
- *`test_X` that contains all the columns of `test_df`*
- *`train_y` that contains the `SalePrice` column of `train_df`*

*Also create the `all_X` variable, using [`pandas.concat`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html?highlight=concat#pandas.concat), to concatenate `train_X` and `test_X`. That will allow us to have an easier time manipulating the two dataframes.*

*__Warning, using `test_X` (and therefore `all_X`) can introduce a statistical bias. You should only use `train_X` when exploring data or to compute a value!__*

In [ ]:
# train_X = ???
# train_y = ???
# test_X = ???
# all_X = ???

### Solution

In [ ]:
train_X = train_df.drop(columns="SalePrice")
train_y = train_df["SalePrice"]
test_X = test_df
all_X = pandas.concat([train_X, test_X])

print("Shape of the training data:", train_X.shape, train_y.shape)
print("Shape of the testing data:", test_X.shape)
print("Shape of the concatenation: ", all_X.shape)

## Data cleaning

This step requires at least 4 substeps:

- handle missing values
- perform specific preprocessing (text, image, …)
- standardize numerical values
- transform categorical values

Here, we will not need specific preprocessing.

### Missing values

Format differences, data gathering conditions and many other factors can lead to missing values.

In this dataset, many columns have `NA` values. Those are not necessarily true missing values, but they require a careful analysis. Because of the arguments we gave to the loading data function, any `NA` value will be considered by pandas as missing here. That will force us to analyze if those `NA`s require a specific processing or not.

- *With the [`pandas.DataFrame.isnull`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isnull.html) function, find out how many values are missing in the training set.*
- *Extend this code to compute the percentage of missing values per column.*
- *Sort the result to show the columns with the most missing values first. You can use [`pandas.Series.sort_values`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.sort_values.html?highlight=sort_values#pandas.DataFrame.sort_values) for that purpose.*
- *Also apply this processing to the test dataframe.*
- *Display all the columns that have missing values, in either set.*

In [ ]:
# Total number of missing values in train_X
# Your code here

In [ ]:
# Percentage of missing values by column in train_X
# Your code here

# Sorted
# Your code here

In [ ]:
# Total number of missing values in test_X
# Your code here

In [ ]:
# Percentage of missing values by column in test_X
# Your code here

# Sorted
# Your code here

In [ ]:
# All the columns with missing values (in either train_X or test_X)
# Your code here

#### Solution

In [ ]:
print("Total number of missing values in train_X:", train_X.isnull().sum().sum())

In [ ]:
def columns_with_missing_values(df: pandas.DataFrame) -> pandas.Series:
  df_na = (df.isnull().sum() / df.shape[0]) * 100
  df_na = df_na[df_na > 0]
  return df_na.sort_values(ascending=False)


def display_percentages_series(s: pandas.Series) -> None:
  print("\n".join(f"{name:>20.20} {value:5.2f}%" for name, value in s.items()))


print("Percentage of missing values by column in train_X:")
missing_train = columns_with_missing_values(train_X)
display_percentages_series(missing_train)

print("Percentage of missing values by column in test_X:")
missing_test = columns_with_missing_values(test_X)
display_percentages_series(missing_test)

In [ ]:
print("Total number of missing values in test_X:", test_X.isnull().sum().sum())

In [ ]:
print("Columns with missing values in either train_X or test_X:")
print(", ".join(set(missing_train.index).union(missing_test.index)))
print()

print("Columns with missing values only in train_X:")
print(", ".join(set(missing_train.index).difference(missing_test.index)))
print()

print("Columns with missing values only in test_X:")
print(", ".join(set(missing_test.index).difference(missing_train.index)))

### Handling missing data

There is no single right way to handle missing data. Each variable must handled in the full context of the study.

*Given the information in `dataset-ames/data_description.txt`, use the `cols_1` to `cols_4` groups to apply 4 different types of missing value handling. All of them should use [`DataFrame.fillna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) that allows to easily fill missing values with the values it's given.*

*You can use, for example:*

- *the mean, the median or 0 for a continuous variable*
- *the mode for a categorical variable (the most present value)*
- *a "NA" class, if it makes sense*
- *…*

*For the exercise, use 1 or 2 columns per group. In the solution, we handle all columns with missing values.*

In [ ]:
# Your code here
cols_1 = ["Alley", "featureX", "..."]
cols_2 = ["SaleType", "featureY", "..."]
# etc…

# all_X[cols_1] = all_X[cols_1].fillna("defaultValue")

#### Solution

In [ ]:
# Median
cols_1 = ["LotFrontage"]
all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

# Mode
cols_2 = [
  "MSZoning",
  "Electrical",
  "KitchenQual",
  "Exterior1st",
  "Exterior2nd",
  "SaleType",
  "Utilities",
]
all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

# 0
cols_4 = [
  "GarageYrBlt",
  "GarageArea",
  "GarageCars",
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtFullBath",
  "BsmtHalfBath",
  "BsmtUnfSF",
  "MasVnrArea",
  "TotalBsmtSF",
]
all_X[cols_4] = all_X[cols_4].fillna(0)

# Specific
cols_5 = ["Functional"]
all_X[cols_5] = all_X[cols_5].fillna("Typ")

# NA
all_X = all_X.fillna("NA")

# Sanity check
print(all_X.isnull().sum().sum())

### Categorical variables encoded as numbers

Some categorical variables can be encoded as numbers in some datasets. That's the case here for the `MSSubClass` feature (see `data_description.txt`). It should be turned into a proper categorical variable for later use by our algorithms.

In [ ]:
# To achieve that, we just have to cast the feature as a string column
cols_numerical2label = ["MSSubClass"]
all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

### Ordinal variables

In order not to lose information for our ordinal variables, we will have to use label encoding as seen during the presentation to transform them for later use.

- *Use label encoding for the `BsmtCond` and `FireplaceQu` variables. You can use the [`DataFrame.replace`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html) function.*

In [ ]:
# all_X=all_X.replace(???)

#### Solution

In [ ]:
replace_mapping = dict(
  Alley=dict(NA=0, Grvl=1, Pave=2),
  BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
  BsmtFinType1=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  BsmtFinType2=dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6),
  Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
  LandSlope=dict(Sev=1, Mod=2, Gtl=3),
  LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
  PavedDrive=dict(NA=0, N=1, P=2, Y=3),
  Street=dict(Grvl=1, Pave=2),
  Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
)

quality_columns = [
  "BsmtCond",
  "BsmtQual",
  "ExterCond",
  "ExterQual",
  "FireplaceQu",
  "GarageCond",
  "GarageQual",
  "HeatingQC",
  "KitchenQual",
  "PoolQC",
]
quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
for quality_column in quality_columns:
  replace_mapping[quality_column] = quality_mapping

all_X.replace(replace_mapping, inplace=True)

### One hot encoding for nominal variables

We only have the nominal variables left.

- *Apply one-hot encoding with the [`pandas.get_dummies`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html) function.*
- *How did the shape of `all_X` evolve?*

In [ ]:
# all_X = ???

#### Solution

In [ ]:
print(all_X.shape)
all_X = pandas.get_dummies(all_X)
print(all_X.shape)

In [ ]:
print("Columns:", ", ".join(all_X.columns))

Each category in the nominal variables is now a column. The input matrix went from $\mathbb{R}^{2919 \times 79}$ to $\mathbb{R}^{2919 \times 245}$.

### Reassembling

Now that the cleaning process is over, we can split `all_X` back into `train_X` and `test_X`.

*Use the shape of `train_X` to know where to split `all_X`, then, perform the split.*

In [ ]:
# split_index = ???
# train_X = ???
# test_X = ???

#### Solution

In [ ]:
split_index = train_X.shape[0]
train_X = all_X.iloc[:split_index, :]
test_X = all_X.iloc[split_index:, :]

## Data exploration

### Target variable analysis

Let's start by analyzing the most important variable: the target variable. This [seaborn doc page](https://seaborn.pydata.org/tutorial/distributions.html#plotting-univariate-distributions) shows the different ways to explore such a variable.

*Use `seaborn` to visualize the output variable.*

In [ ]:
# Your code here

#### Solution

In [ ]:
seaborn.displot(train_y, kde=True)
plt.show()

### Comparing the target distribution to a normal distribution

A prerequisite of many approaches is to have a normally distributed target variable.

*Test this hypothesis using for example [`scipy.stats.probplot`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.probplot.html).*

In [ ]:
# Your code here

#### Solution

In [ ]:
seaborn.distplot(train_y, fit=scipy.stats.norm)
plt.show()

fig = plt.figure()
scipy.stats.probplot(train_y, dist="norm", plot=plt)
plt.show()

### Correlations to the target variable

Now that we have a good idea of the target variable shape, let's study the variables that are most correlated to it.

*Use [`DataFrame.corr`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) or [`DataFrame.corrwith`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html) and [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) to display a correlation matrix. If possible, sort it by correlation with the target.*

In [ ]:
# Your code here

#### Solution

In [ ]:
target_corrs = (
  train_X
  # Correlations with the target
  .corrwith(train_y)
  # Delete NaNs: they appear because of constant columns
  .dropna()
  # Sort by correlation with the target
  .sort_values(ascending=False)
)

# Get the top 10 corrs and anti-corrs
top_target_corrs = pandas.concat([target_corrs[:10], target_corrs[-10:]])
top_target_corrs.rename("SalePrice", inplace=True)

# Compute the correlation matrix
feature_corrs = train_X.loc[:, top_target_corrs.index].corr()

# Join target correlations with feature correlations
corrs = pandas.concat([top_target_corrs, feature_corrs], axis=1)

# Display with a heatmap
_, ax = plt.subplots(figsize=(10, 6))
seaborn.heatmap(corrs, vmin=-1, vmax=1, cmap=seaborn.diverging_palette(220, 20, n=100))
ax.set_title("Plus grandes corrélations et anti-corrélations")
plt.show()

### Exploration of the most correlated variables

*Plot the joint distribution of `GrLivArea` and `SalePrice`. Do the same for `OverallQual` and `SalePrice`.*

In [ ]:
# Your code here

#### Solution

In [ ]:
seaborn.jointplot(x=train_X["GrLivArea"], y=train_y)
plt.show()
seaborn.violinplot(x=train_X["OverallQual"], y=train_y)
plt.show()

### Analysis of the most correlated variables

*What do you notice on those two plots?*

#### Answer

We can see two outliers in the `GrLivArea` plot.

We need a deeper analysis to determine if those points should be kept or if they should be removed. How would you proceed?